# PA2 Task 7: pairwise judging on Kaggle

Create this notebook from https://www.kaggle.com/benchmarks/tasks/new so `kaggle_benchmarks` and the model proxy are configured.

1. Attach the dataset that holds the pairwise files (`*_pairs.jsonl`) produced on Modal.
2. Run all cells. Judgments are appended after every batch to `/kaggle/working/eval/`, so a rerun resumes.
3. Save a notebook version, then copy `*_judged.jsonl`, `*_failures.jsonl`, `*_summary.json` back into `outputs/eval/full/` in the repo.

In [ ]:
import glob, os, subprocess, sys
REPO_URL = 'https://github.com/Musa-Gondal/Advgenai-pa2.git'  # or attach the repo as a dataset if it is private
BRANCH = 'main'
if not os.path.exists('/kaggle/working/genai-pa2'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO_URL, '/kaggle/working/genai-pa2'], check=True)
SCRIPT = '/kaggle/working/genai-pa2/scripts/judge_pairwise_kaggle.py'
PAIRS = sorted(glob.glob('/kaggle/input/**/*_pairs.jsonl', recursive=True))
print(PAIRS)

In [ ]:
import kaggle_benchmarks as kbench
JUDGE = 'google/gemini-3.6-flash'
print('available:', sorted(kbench.llms))
if JUDGE not in kbench.llms:
    print(f'{JUDGE} not listed; pick another judge from the list above and record it in the report')

In [ ]:
# Quick check on 10 pairs of the first file before the full run.
os.makedirs('/kaggle/working/eval', exist_ok=True)
p = PAIRS[0]; name = os.path.basename(p).replace('_pairs.jsonl', '')
subprocess.run([sys.executable, SCRIPT, '--pairs_path', p, '--output_path', f'/kaggle/working/eval/{name}_trial_judged.jsonl',
                '--judge_model', JUDGE, '--limit', '10', '--batch_size', '5', '--sleep_seconds', '5', '--max_retries', '3'], check=True)

In [ ]:
for p in PAIRS:
    name = os.path.basename(p).replace('_pairs.jsonl', '')
    print('=' * 20, name)
    subprocess.run([sys.executable, SCRIPT, '--pairs_path', p, '--output_path', f'/kaggle/working/eval/{name}_judged.jsonl',
                    '--judge_model', JUDGE, '--batch_size', '5', '--sleep_seconds', '5', '--max_retries', '3',
                    '--reuse_judgments'], check=True)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/judgments', 'zip', '/kaggle/working/eval')
print(os.listdir('/kaggle/working/eval'))